# Task 2. CLAP Embeddings & Dimensionality Reduction

This notebook demonstrates how to compute CLAP embeddings for audio files, segment them, and visualize the results using PCA and t-SNE.

Before doing anything, let's install dependencies. You may need to restart the kernel after installing everything.

In [ ]:
# Install dependencies
!pip install torch torchaudio librosa numpy matplotlib tqdm msclap scikit-learn

The following cell contains a wrapper that can be used to run CLAP with a simple syntax. Run the cell so we can use it. From this point, it is completely fine to ask any LLM what this code is actually doing.

In [ ]:
import librosa
import torch
import numpy as np
import torch.nn.functional as F

from msclap.CLAPWrapper import CLAPWrapper

# Clap wrapper with custom audio processing for embedding computation of short audio signals
class CLAPWrapper(CLAPWrapper):
    def __init__(self, version="2023", use_cuda=False):
        super().__init__(version=version, use_cuda=use_cuda)

        self.name = "CLAPWrapper"

    def compute_embedding(self, audio, og_sr=48000, random_extension=True):
        # Check that dimensions are B x C x T
        if audio.dim() != 3:
            raise ValueError(f"Expected audio tensor to have 3 dimensions (Batch, Channels, Time), but got {audio.shape}")
        clap_sr = self.args.sampling_rate
        # Resampling + mono if needed
        audio = resample_and_mono_audio(audio, og_sr, clap_sr, mono=True)
        # Extend/trim to 8 seconds
        audio = audio_extender(audio, 
                            random_extension=random_extension,
                            sample_rate=clap_sr, 
                            duration=8.0, 
                            overlap_ratio=0.1) # [B, 1, 8*48000]
        embedding = self._get_audio_embeddings(audio) # [B, 1024]

        # Normalize embeddings to unit norm
        embedding = F.normalize(embedding, p=2, dim=1)
        return embedding

def resample_and_mono_audio(audio, og_sr, target_sr, mono=True):
    """
    Resample audio from og_sr → target_sr.

    Args:
        audio: Tensor, shape (C, T) or (B, C, T)
        og_sr: original sample rate
        target_sr: target sample rate
        batch_added: bool, if True, squeeze batch dim at the end

    Returns:
        resampled audio tensor
        shape: (C, T_new) or (B, C, T_new) depending on batch_added
    """

    # --- Ensure batch dimension ---
    if audio.dim() == 2:
        audio = audio.unsqueeze(0)  # (1, C, T)
        added_batch = True
    else:
        added_batch = False

    # --- Convert to mono if needed ---
    if mono and audio.shape[1] > 1:
        audio = audio.mean(dim=1, keepdim=True)  # (B, 1, T)

    B, C, T = audio.shape
    T_new = int(T * target_sr / og_sr)

    # Interpolate along time dimension
    # F.interpolate expects (B, C, L) with float, uses mode='linear'
    audio = audio.float()
    resampled = F.interpolate(audio, size=T_new, mode='linear', align_corners=False)

    # Remove batch if needed
    if added_batch:
        resampled = resampled.squeeze(0)

    return resampled

def audio_extender(input_audio, random_extension=False, 
                   sample_rate=48000, duration=7.0, overlap_ratio=0.1):
    """
    Extends or trims audio to a fixed duration with optional random segmenting.

    Args:
        input_audio: Tensor (C, T) or (B, C, T)
        random_extension: bool
        sample_rate: int
        duration: float (seconds)
        overlap_ratio: float (default 0.1 → 100ms at 48kHz)

    Returns:
        Tensor of shape (B, 1, expected_length)
    """

    # --- Ensure batch dimension ---
    if input_audio.dim() == 2:
        input_audio = input_audio.unsqueeze(0)  # (1, C, T)
        batch_added = True
    else:        
        batch_added = False

    B, C, T = input_audio.shape

    # --- Convert to mono ---
    audio = input_audio.mean(dim=1, keepdim=True)  # (B, 1, T)

    expected_length = int(sample_rate * duration)
    overlap = int(overlap_ratio * sample_rate)

    # --- Adjust overlap if audio is shorter than overlap window ---
    overlap = min(overlap, T // 2)  # ensure safe crossfade

    # --- If longer → trim ---
    if T >= expected_length:
        return audio[:, :, :expected_length]

    # --- Create output tensor ---
    output = torch.zeros((B, 1, expected_length), device=audio.device)

    # --- Crossfade windows ---
    fade_out = torch.linspace(1, 0, overlap, device=audio.device)
    fade_in  = torch.linspace(0, 1, overlap, device=audio.device)

    pos = 0
    first_segment = True

    while pos < expected_length:
        remaining = expected_length - pos

        # === NON RANDOM MODE ===
        if not random_extension:
            segment = audio
        # === RANDOM MODE ===
        else:
            min_length = T // 2
            max_length = T
            segment_length = np.random.randint(min_length, max_length + 1)

            if T > segment_length:
                start_pos = np.random.randint(0, T - segment_length + 1)
            else:
                start_pos = 0
                segment_length = T

            segment = audio[:, :, start_pos:start_pos + segment_length]

        seg_len = segment.shape[-1]

        # --- First segment: no fade-in ---
        if first_segment:
            use_length = min(seg_len, remaining)
            output[:, :, pos:pos + use_length] = segment[:, :, :use_length]
            pos += use_length
            first_segment = False
            continue

        # --- Crossfade segment ---
        if remaining <= overlap:
            # Last tiny bit
            output[:, :, pos:pos + remaining] = (
                output[:, :, pos:pos + remaining] * fade_out[:remaining] +
                segment[:, :, :remaining] * fade_in[:remaining]
            )
            break

        # Overlap region
        output[:, :, pos:pos + overlap] = (
            output[:, :, pos:pos + overlap] * fade_out +
            segment[:, :, :overlap] * fade_in
        )

        # Non-overlap region
        non_overlap = min(seg_len - overlap, remaining - overlap)
        if non_overlap > 0:
            output[:, :, pos + overlap:pos + overlap + non_overlap] = \
                segment[:, :, overlap:overlap + non_overlap]

        pos += non_overlap

    # --- Normalize ---
    max_val = torch.max(torch.abs(output))
    if max_val > 0:
        output = output / max_val

    if batch_added:
        output = output.squeeze(0)  # (1, 1, T) → (1, T)

    return output


## Simple Example: Compute a CLAP Embedding

This cell shows how to compute a CLAP embedding for a single audio file.

In [ ]:
# Simple example: compute CLAP embedding for an audio file
clap = CLAPWrapper(use_cuda=torch.cuda.is_available())
device = clap.device if hasattr(clap, 'device') else ('cuda' if torch.cuda.is_available() else 'cpu')

audio_path = "path/to/your/audio/file.wav"  # <- Change this to your file

waveform, sr = librosa.load(audio_path, sr=48000, mono=True)
tensor_audio = torch.tensor(waveform).unsqueeze(0).unsqueeze(0)  # [B, C, T]
tensor_audio = tensor_audio.to(device)

embedding = clap.compute_embedding(tensor_audio, og_sr=sr)
print("CLAP embedding shape:", embedding.shape)
print("CLAP embedding (first 5 values):", embedding[0, :5])

## Segment Audio Files and Build Dataset

This cell segments each audio file into 1-second pieces, computes CLAP embeddings for each segment, and builds a dataset with labels for each embedding (the file stem). It also prints a summary of the dataset.

In [ ]:
# Segment audio files, compute CLAP embeddings, and build dataset
import os
from tqdm import tqdm

def segment_audio(waveform, sr, segment_sec=1.0):
    seg_len = int(sr * segment_sec)
    total_len = waveform.shape[-1]
    n_segs = total_len // seg_len
    segments = [waveform[..., i*seg_len:(i+1)*seg_len] for i in range(n_segs)]
    return segments

# List of audio paths (replace with your own)
audio_paths = ["/path/to/your/audio/file1.wav", 
               "/path/to/your/audio/file2.wav"] # <- Change these to your files. You can add more!

clap = CLAPWrapper(use_cuda=torch.cuda.is_available())
embeddings = []
labels = []
segment_counts = []
file_lengths = []

for path in tqdm(audio_paths, desc="Processing files"):
    y, sr = librosa.load(path, sr=48000, mono=True)
    file_lengths.append(len(y) / sr)
    segments = segment_audio(torch.tensor(y).unsqueeze(0), sr)
    if len(segments) == 0:
        continue
    segment_counts.append(len(segments))
    for seg in segments:
        seg = seg.unsqueeze(0).to(device)  # [B, C, T]
        emb = clap.compute_embedding(seg, og_sr=sr)
        embeddings.append(emb.squeeze(0).cpu().numpy())
        labels.append(os.path.splitext(os.path.basename(path))[0])

embeddings = np.stack(embeddings)
labels = np.array(labels)

print(f"Total segments: {len(embeddings)}")
print(f"Files processed: {len(audio_paths)}")
print(f"Segments per file: {segment_counts}")
print(f"File durations (s): {file_lengths}")
print(f"Labels example: {labels[:5]}")

## Dimensionality Reduction Wrapper

This cell provides a simple wrapper for dimensionality reduction using PCA or t-SNE, and visualization of the results.

In [ ]:
# Dimensionality reduction wrapper (PCA & t-SNE)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def reduce_and_plot(embeddings, labels, method="pca", show_legend=True):
    if method == "pca":
        reducer = PCA(n_components=2)
    elif method in ["tsne", "t-sne"]:
        reducer = TSNE(n_components=2, perplexity=30, random_state=42)
    else:
        raise ValueError("Method must be 'pca' or 'tsne'")
    reduced = reducer.fit_transform(embeddings)
    unique_labels = sorted(set(labels))
    cmap = plt.get_cmap("tab10" if len(unique_labels) <= 10 else "tab20")
    label_to_color = {lbl: cmap(i % cmap.N) for i, lbl in enumerate(unique_labels)}
    colors = [label_to_color[lbl] for lbl in labels]
    plt.figure(figsize=(8, 6))
    plt.scatter(reduced[:, 0], reduced[:, 1], c=colors, alpha=0.7, s=15, edgecolors='none')
    plt.title(f"CLAP Embeddings ({method.upper()})")
    plt.grid(True, alpha=0.3)
    if show_legend and len(unique_labels) <= 30:
        from matplotlib.lines import Line2D
        legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor=label_to_color[lbl], markersize=8, label=lbl) for lbl in unique_labels]
        plt.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1.05, 0.5), fontsize=9)
    elif show_legend:
        print(f"Note: Suppressed legend because there are too many unique files ({len(unique_labels)}).")
    plt.tight_layout()
    plt.show()

## Visualize with PCA

This cell computes and visualizes PCA for your dataset.

In [ ]:
# Visualize with PCA
reduce_and_plot(embeddings, labels, method="pca")

In [ ]:
# Visualize with t-SNE
reduce_and_plot(embeddings, labels, method="tsne")